# 08 Advanced Challenge — Data QC and Outlier Visualization in Python

## Goal

Use visual checks to identify suspicious points in synthetic assay data.

This introduces a real-world style idea: data quality matters before interpretation.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px

df = pd.read_csv("../data/qc_outlier_sample.csv")
df.head()

In [ ]:
# Compute z-score within each drug/concentration group.
df["group_key"] = df["drug_name"] + "_" + df["concentration_uM"].astype(str)

df["group_mean"] = df.groupby("group_key")["cell_viability_percent"].transform("mean")
df["group_sd"] = df.groupby("group_key")["cell_viability_percent"].transform("std")
df["z_score"] = (df["cell_viability_percent"] - df["group_mean"]) / df["group_sd"]

df["qc_flag"] = np.where(df["z_score"].abs() > 1.5, "review", "ok")
df

In [ ]:
fig = px.scatter(
    df,
    x="concentration_uM",
    y="cell_viability_percent",
    color="qc_flag",
    symbol="drug_name",
    hover_data=["sample_id", "drug_name", "replicate", "z_score"],
    title="QC Scatter Plot: Possible Outliers"
)
fig.update_xaxes(type="log")
fig.show()

In [ ]:
fig2 = px.box(
    df,
    x="drug_name",
    y="cell_viability_percent",
    color="drug_name",
    points="all",
    title="Cell Viability Distribution by Drug"
)
fig2.show()

## Challenge Questions

1. Which points were flagged for review?
2. Are flagged points always wrong?
3. What should a lab scientist do before removing an outlier?
4. How could this QC view improve BioDose AI?